# PD Model – Paper-Grade Validation (02a)

**Purpose:** Run validation analyses expected by credit-risk and ML reviewers: **temporally-aware CV** (walk-forward / expanding window), **bootstrapped AUC confidence intervals**, **calibration** (Brier, reliability diagram), and **Population Stability Index (PSI)** per feature between train and OOT. Uses the same data and split as `02a_pd_xgboost_training.ipynb`. No retraining of the main model; loads existing artifacts when needed for calibration/PSI.

**Prerequisites:** Run `01_pd_lendingclub_feature_engineering.ipynb` and `02a_pd_xgboost_training.ipynb`. For true temporal ordering within the training window, the engineered parquet should include `issue_year`; otherwise row order is used as a proxy.

## 1. Load data (same as 02a)

In [ ]:
# Extends 02a_pd_xgboost_training.ipynb with publication-grade validation
# Walk-forward CV, bootstrapped AUC CIs, calibration, PSI per feature
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
for _ in range(5):
    if (ROOT / "credit_risk").is_dir() and (ROOT / "data").is_dir():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from credit_risk.feature_engineering.common_features import get_feature_names_no_leakage_v2
from credit_risk.feature_engineering.feature_screening import screen_features_train_only

DATA_PATH = ROOT / "data" / "credit_risk_pd" / "LendingClub" / "processed" / "lendingclub_engineered.parquet"
df = pd.read_parquet(DATA_PATH)
all_feature_names = get_feature_names_no_leakage_v2()
X = df[[c for c in all_feature_names if c in df.columns]].copy()
for c in all_feature_names:
    if c not in X.columns:
        X[c] = 0.0
X = X[all_feature_names]
y = df["default"]

train_idx = df["split"] == "train"
val_idx = df["split"] == "val"
test_idx = df["split"] == "test"
X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

screening = screen_features_train_only(X_train, y_train, missingness_threshold=0.50, min_ks=0.001, corr_threshold=0.95)
feature_names = screening.selected_features
X_train = X_train[feature_names]
X_val = X_val[feature_names]
X_test = X_test[feature_names]
medians = X_train.median()
X_val_filled = X_val.fillna(medians)
X_test_filled = X_test.fillna(medians)

print(f"Train {X_train.shape[0]:,} / Val {X_val.shape[0]:,} / Test {X_test.shape[0]:,} | {len(feature_names)} features")

## 2. Temporal order for walk-forward CV

If the parquet has `issue_year` (or `issue_d`), use it to order training samples; otherwise assume row order preserves time (as in notebook 01).

In [ ]:
if "issue_year" in df.columns:
    train_order = np.argsort(df.loc[train_idx, "issue_year"].values)
    time_label = "issue_year"
elif "issue_d" in df.columns:
    from datetime import datetime
    def parse_issue_d(s):
        try:
            return datetime.strptime(str(s).strip(), "%b-%Y")
        except Exception:
            return None
    issue_dt = df.loc[train_idx, "issue_d"].apply(parse_issue_d)
    train_order = np.argsort(issue_dt.fillna(pd.Timestamp.min).astype(np.int64))
    time_label = "issue_d"
else:
    train_order = np.arange(X_train.shape[0])  # assume row order = time order
    time_label = "row order"
X_train_ordered = X_train.iloc[train_order].reset_index(drop=True)
y_train_ordered = y_train.iloc[train_order].reset_index(drop=True)
print(f"Temporal ordering: {time_label}")

## 3. Walk-forward / expanding-window CV

Within the training window, use **expanding window** folds: fold k trains on the first k segments and validates on segment k+1. No shuffle; order is strictly temporal.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

N_FOLDS = 5
n = len(X_train_ordered)
fold_sizes = np.array([n // N_FOLDS] * N_FOLDS)
fold_sizes[: n % N_FOLDS] += 1
ends = np.cumsum(fold_sizes)
starts = np.concatenate([[0], ends[:-1]])

cv_aucs = []
for k in range(N_FOLDS - 1):
    tr_end = ends[k]
    va_start, va_end = starts[k + 1], ends[k + 1]
    X_tr = X_train_ordered.iloc[:tr_end].fillna(medians)
    y_tr = y_train_ordered.iloc[:tr_end]
    X_va = X_train_ordered.iloc[va_start:va_end].fillna(medians)
    y_va = y_train_ordered.iloc[va_start:va_end]
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    lr = LogisticRegression(max_iter=500, random_state=42, class_weight="balanced")
    lr.fit(X_tr_s, y_tr)
    p_va = lr.predict_proba(X_va_s)[:, 1]
    auc_k = roc_auc_score(y_va, p_va)
    cv_aucs.append(auc_k)

print("Expanding-window CV (LR proxy):", cv_aucs)
print("Mean AUC = {:.4f}, std = {:.4f}".format(np.mean(cv_aucs), np.std(cv_aucs)))

## 4. Bootstrapped AUC confidence intervals (OOT test)

Bootstrap the test set to get AUC distribution and 95% CI so the paper can report stability, not just a single point estimate.

In [ ]:
import joblib
from sklearn.metrics import roc_auc_score

MODEL_DIR = ROOT / "models" / "pd"
rng = np.random.default_rng(42)
N_BOOT = 1000
n_test = len(y_test)

p_test = None
if (MODEL_DIR / "pd_model_local_v2.pkl").exists():
    try:
        import credit_risk.models.pd_model
        data = joblib.load(MODEL_DIR / "pd_model_local_v2.pkl")
        model = data["model"]
        p_test = model.predict_proba(X_test_filled)[:, 1]
    except Exception as e:
        print("Load v2 failed:", e)
if p_test is None and (MODEL_DIR / "pd_model_local_v1.pkl").exists():
    data = joblib.load(MODEL_DIR / "pd_model_local_v1.pkl")
    p_test = data["model"].predict_proba(X_test_filled)[:, 1]

if p_test is not None:
    auc_boot = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, n_test, size=n_test)
        auc_boot.append(roc_auc_score(y_test.iloc[idx], p_test[idx]))
    auc_boot = np.array(auc_boot)
    ci_lo, ci_hi = np.percentile(auc_boot, [2.5, 97.5])
    print("OOT test AUC: {:.4f} [95% CI: {:.4f} – {:.4f}] (bootstrap n={})".format(
        np.mean(auc_boot), ci_lo, ci_hi, N_BOOT))
else:
    print("No saved model found; run 02a first.")

## 5. Calibration: Brier score and reliability diagram

AUC measures discrimination; credit models also need **calibration** (predicted PD ≈ actual default rate). I report Brier score and a reliability diagram (mean predicted vs mean actual in bins).

In [ ]:
from sklearn.metrics import brier_score_loss
import matplotlib.pyplot as plt

if p_test is not None:
    brier = brier_score_loss(y_test, p_test)
    print("Brier score (lower is better): {:.4f}".format(brier))

    n_bins = 10
    bins = np.linspace(0, 1, n_bins + 1)
    bin_inds = np.clip(np.digitize(p_test, bins) - 1, 0, n_bins - 1)
    mean_pred = []
    mean_actual = []
    for j in range(n_bins):
        m = bin_inds == j
        if m.sum() > 0:
            mean_pred.append(p_test[m].mean())
            mean_actual.append(y_test.iloc[m].mean())
        else:
            mean_pred.append(np.nan)
            mean_actual.append(np.nan)
    mean_pred = np.array(mean_pred)
    mean_actual = np.array(mean_actual)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
    ax.scatter(mean_pred, mean_actual, s=60, label="Model")
    ax.set_xlabel("Mean predicted PD")
    ax.set_ylabel("Mean actual default rate")
    ax.set_title("Reliability diagram (OOT test)")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No predictions; run model load cell first.")

## 6. Population Stability Index (PSI) per feature

PSI measures distributional shift between train and OOT test. Standard in credit model validation; report per-feature PSI so reviewers see which inputs shifted most.

In [ ]:
def psi_single(x_train, x_test, n_bins=10):
    breakpoints = np.nanpercentile(x_train, np.linspace(0, 100, n_bins + 1))
    breakpoints[-1] += 1e-9
    breakpoints = np.unique(breakpoints)
    if len(breakpoints) < 2:
        return 0.0
    p_train = np.histogram(x_train, breakpoints)[0] / len(x_train)
    p_test = np.histogram(x_test, breakpoints)[0] / len(x_test)
    p_train = np.clip(p_train, 1e-6, 1)
    p_test = np.clip(p_test, 1e-6, 1)
    return np.sum((p_test - p_train) * np.log(p_test / p_train))

psi_by_feature = {}
for col in feature_names:
    xt = X_train[col].values.astype(float)
    xtest = X_test[col].values.astype(float)
    xt, xtest = np.nan_to_num(xt, nan=np.nanmedian(xt)), np.nan_to_num(xtest, nan=np.nanmedian(xt))
    psi_by_feature[col] = psi_single(xt, xtest)

psi_series = pd.Series(psi_by_feature).sort_values(ascending=False)
print("PSI (train vs OOT test) – top 15 features:")
print(psi_series.head(15).to_string())
print("\nInterpretation: PSI < 0.1 stable; 0.1–0.25 moderate shift; > 0.25 large shift.")